# Apache Iceberg com Apache Spark

## Cenário: Loja Virtual

Este notebook demonstra o uso do **Apache Iceberg** integrado ao **Apache Spark (PySpark)** para gerenciar dados de uma **Loja Virtual**.

---

## Modelo de Dados (ER)

![Diagrama Entidade-Relacionamento](../docs/assets/er_diagram.png)

## Fonte de Dados

Dados **sintéticos** gerados diretamente no notebook, representando uma loja virtual com clientes, produtos e pedidos.

---

## DDL (Schema das Tabelas)

```sql
-- Namespace Iceberg
CREATE NAMESPACE IF NOT EXISTS local.loja;

-- Tabela de Clientes
CREATE TABLE IF NOT EXISTS local.loja.clientes (
    id     INT, nome STRING, email STRING, cidade STRING
) USING iceberg;

-- Tabela de Produtos
CREATE TABLE IF NOT EXISTS local.loja.produtos (
    id INT, nome STRING, categoria STRING, preco DOUBLE, estoque INT
) USING iceberg;

-- Tabela de Pedidos
CREATE TABLE IF NOT EXISTS local.loja.pedidos (
    id INT, cliente_id INT, produto_id INT, quantidade INT,
    valor_total DOUBLE, data_pedido STRING, status STRING
) USING iceberg;
```

## 1. Configuração do Ambiente

In [1]:
import os
import sys
import shutil

# Garante que o Spark usa o Python do ambiente virtual
python_path = sys.executable
os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path
os.environ["HADOOP_HOME"] = "C:\\hadoop"
# Carrega hadoop.dll para suporte a operações de arquivo no Windows
os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.library.path=C:\\hadoop\\bin"

from pyspark.sql import SparkSession

# Versão do Iceberg runtime compatível com Spark 3.5
ICEBERG_JAR = "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3"

# Diretório do warehouse Iceberg
WAREHOUSE = os.path.abspath("./iceberg-warehouse").replace("\\", "/")

# Limpa warehouse anterior para execução limpa
shutil.rmtree("./iceberg-warehouse", ignore_errors=True)

spark = (
    SparkSession.builder
    .appName("Apache Iceberg - Loja Virtual")
    .config("spark.jars.packages", ICEBERG_JAR)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", WAREHOUSE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"PySpark {spark.version} iniciado com Apache Iceberg!")
print(f"Warehouse: {WAREHOUSE}")

PySpark 3.5.0 iniciado com Apache Iceberg!
Warehouse: c:/Users/aliss_16u0t6a/Downloads/TrabalhoEngDados/trabalho-apachespark/iceberg/iceberg-warehouse


## 2. Criação do Namespace e Tabelas Iceberg (DDL)

In [2]:
# Cria o namespace (equivalente ao schema/banco de dados)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.loja")

# DDL - Tabela de Clientes
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.loja.clientes (
        id     INT,
        nome   STRING,
        email  STRING,
        cidade STRING
    ) USING iceberg
""")

# DDL - Tabela de Produtos
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.loja.produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     DOUBLE,
        estoque   INT
    ) USING iceberg
""")

# DDL - Tabela de Pedidos
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.loja.pedidos (
        id          INT,
        cliente_id  INT,
        produto_id  INT,
        quantidade  INT,
        valor_total DOUBLE,
        data_pedido STRING,
        status      STRING
    ) USING iceberg
""")

print("Tabelas Iceberg criadas com sucesso!")
spark.sql("SHOW TABLES IN local.loja").show()

Tabelas Iceberg criadas com sucesso!
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     loja| produtos|      false|
|     loja| clientes|      false|
|     loja|  pedidos|      false|
+---------+---------+-----------+



## 3. INSERT — Inserindo Dados nas Tabelas

In [3]:
# INSERT - Clientes
spark.sql("""
    INSERT INTO local.loja.clientes VALUES
        (1, 'Ana Souza',    'ana@email.com',    'Florianopolis'),
        (2, 'Bruno Lima',   'bruno@email.com',  'Criciuma'),
        (3, 'Carla Mendes', 'carla@email.com',  'Joinville'),
        (4, 'Diego Ramos',  'diego@email.com',  'Blumenau'),
        (5, 'Elena Costa',  'elena@email.com',  'Chapeco')
""")

# INSERT - Produtos
spark.sql("""
    INSERT INTO local.loja.produtos VALUES
        (1, 'Notebook Dell',    'Eletronicos', 3500.0, 50),
        (2, 'Mouse Logitech',   'Perifericos',   80.0, 200),
        (3, 'Teclado Mecanico', 'Perifericos',  150.0, 120),
        (4, 'Monitor 24pol',    'Eletronicos', 1200.0,  75),
        (5, 'Headset Gaming',   'Perifericos',  250.0,  90)
""")

# INSERT - Pedidos
spark.sql("""
    INSERT INTO local.loja.pedidos VALUES
        (1, 1, 1, 1, 3500.0, '2024-01-10', 'entregue'),
        (2, 2, 2, 2,  160.0, '2024-01-12', 'entregue'),
        (3, 3, 3, 1,  150.0, '2024-01-15', 'em_transito'),
        (4, 4, 4, 1, 1200.0, '2024-01-18', 'processando'),
        (5, 5, 5, 2,  500.0, '2024-01-20', 'processando')
""")

print("=== Clientes ===")
spark.sql("SELECT * FROM local.loja.clientes ORDER BY id").show()

print("=== Produtos ===")
spark.sql("SELECT * FROM local.loja.produtos ORDER BY id").show()

print("=== Pedidos ===")
spark.sql("SELECT * FROM local.loja.pedidos ORDER BY id").show()

=== Clientes ===
+---+------------+---------------+-------------+
| id|        nome|          email|       cidade|
+---+------------+---------------+-------------+
|  1|   Ana Souza|  ana@email.com|Florianopolis|
|  2|  Bruno Lima|bruno@email.com|     Criciuma|
|  3|Carla Mendes|carla@email.com|    Joinville|
|  4| Diego Ramos|diego@email.com|     Blumenau|
|  5| Elena Costa|elena@email.com|      Chapeco|
+---+------------+---------------+-------------+

=== Produtos ===
+---+----------------+-----------+------+-------+
| id|            nome|  categoria| preco|estoque|
+---+----------------+-----------+------+-------+
|  1|   Notebook Dell|Eletronicos|3500.0|     50|
|  2|  Mouse Logitech|Perifericos|  80.0|    200|
|  3|Teclado Mecanico|Perifericos| 150.0|    120|
|  4|   Monitor 24pol|Eletronicos|1200.0|     75|
|  5|  Headset Gaming|Perifericos| 250.0|     90|
+---+----------------+-----------+------+-------+

=== Pedidos ===
+---+----------+----------+----------+-----------+-------

## 4. UPDATE — Atualizando Dados

In [4]:
# UPDATE - Reajuste de 10% nos preços de Eletrônicos
spark.sql("""
    UPDATE local.loja.produtos
    SET preco = preco * 1.10
    WHERE categoria = 'Eletronicos'
""")

# UPDATE - Marca pedidos entregues como finalizados
spark.sql("""
    UPDATE local.loja.pedidos
    SET status = 'finalizado'
    WHERE status = 'entregue'
""")

print("=== Produtos após UPDATE (preco Eletronicos +10%) ===")
spark.sql("SELECT * FROM local.loja.produtos ORDER BY id").show()

print("=== Pedidos após UPDATE (entregue -> finalizado) ===")
spark.sql("SELECT * FROM local.loja.pedidos ORDER BY id").show()

=== Produtos após UPDATE (preco Eletronicos +10%) ===
+---+----------------+-----------+------------------+-------+
| id|            nome|  categoria|             preco|estoque|
+---+----------------+-----------+------------------+-------+
|  1|   Notebook Dell|Eletronicos|3850.0000000000005|     50|
|  2|  Mouse Logitech|Perifericos|              80.0|    200|
|  3|Teclado Mecanico|Perifericos|             150.0|    120|
|  4|   Monitor 24pol|Eletronicos|            1320.0|     75|
|  5|  Headset Gaming|Perifericos|             250.0|     90|
+---+----------------+-----------+------------------+-------+

=== Pedidos após UPDATE (entregue -> finalizado) ===
+---+----------+----------+----------+-----------+-----------+-----------+
| id|cliente_id|produto_id|quantidade|valor_total|data_pedido|     status|
+---+----------+----------+----------+-----------+-----------+-----------+
|  1|         1|         1|         1|     3500.0| 2024-01-10| finalizado|
|  2|         2|         2|       

## 5. DELETE — Removendo Dados

In [5]:
# Zera estoque do produto 5 para demonstrar DELETE
spark.sql("UPDATE local.loja.produtos SET estoque = 0 WHERE id = 5")

# DELETE - Remove produtos sem estoque
spark.sql("""
    DELETE FROM local.loja.produtos
    WHERE estoque = 0
""")

print("=== Produtos após DELETE (sem estoque removidos) ===")
spark.sql("SELECT * FROM local.loja.produtos ORDER BY id").show()

=== Produtos após DELETE (sem estoque removidos) ===
+---+----------------+-----------+------------------+-------+
| id|            nome|  categoria|             preco|estoque|
+---+----------------+-----------+------------------+-------+
|  1|   Notebook Dell|Eletronicos|3850.0000000000005|     50|
|  2|  Mouse Logitech|Perifericos|              80.0|    200|
|  3|Teclado Mecanico|Perifericos|             150.0|    120|
|  4|   Monitor 24pol|Eletronicos|            1320.0|     75|
+---+----------------+-----------+------------------+-------+



## 6. Consultas Analíticas

In [6]:
# Total de vendas por status
print("=== Total de vendas por status ===")
spark.sql("""
    SELECT status,
           COUNT(*) AS qtd_pedidos,
           SUM(valor_total) AS total_vendas
    FROM local.loja.pedidos
    GROUP BY status
    ORDER BY total_vendas DESC
""").show()

# Pedidos com nome do cliente e produto
print("=== Pedidos com detalhes ===")
spark.sql("""
    SELECT c.nome AS cliente, p.nome AS produto,
           ped.quantidade, ped.valor_total, ped.status
    FROM local.loja.pedidos ped
    JOIN local.loja.clientes c ON ped.cliente_id = c.id
    JOIN local.loja.produtos p ON ped.produto_id = p.id
    ORDER BY ped.id
""").show()

spark.stop()
print("Sessão Spark encerrada.")

=== Total de vendas por status ===
+-----------+-----------+------------+
|     status|qtd_pedidos|total_vendas|
+-----------+-----------+------------+
| finalizado|          2|      3660.0|
|processando|          2|      1700.0|
|em_transito|          1|       150.0|
+-----------+-----------+------------+

=== Pedidos com detalhes ===
+------------+----------------+----------+-----------+-----------+
|     cliente|         produto|quantidade|valor_total|     status|
+------------+----------------+----------+-----------+-----------+
|   Ana Souza|   Notebook Dell|         1|     3500.0| finalizado|
|  Bruno Lima|  Mouse Logitech|         2|      160.0| finalizado|
|Carla Mendes|Teclado Mecanico|         1|      150.0|em_transito|
| Diego Ramos|   Monitor 24pol|         1|     1200.0|processando|
+------------+----------------+----------+-----------+-----------+

Sessão Spark encerrada.
